<a href="https://colab.research.google.com/github/Beluncho/AI_Agro_cons_sale/blob/main/BackendAgroSale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# ПОЛНЫЙ БЭКЕНД С ПРОКСИ ДЛЯ КАРТИНОК
# КОПИРУЙ И ВСТАВЛЯЙ В COLAB
# ============================================

# ---------- 0) Установка ----------
!pip -q install fastapi "uvicorn[standard]" aiohttp beautifulsoup4 lxml
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# ---------- 1) Код бэкенда ----------
backend_code = r'''
import json
import os
import re
import asyncio
import xml.etree.ElementTree as ET
from typing import List, Dict, Optional
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse, Response
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import aiohttp
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

app = FastAPI(title="Pkyar Production Parser")

# CORS - разрешаем все запросы
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

DATA_DIR = "data"
PRODUCTION_FILE = os.path.join(DATA_DIR, "production.json")
CHUNK_SIZE = 800

os.makedirs(DATA_DIR, exist_ok=True)

# Модели данных
class ProductChunk(BaseModel):
    id: str
    title: str
    text: str
    image: str
    features: Dict[str, str]

class ParseStatus(BaseModel):
    is_parsing: bool = False
    total_products: int = 0
    parsed_products: int = 0
    current_stage: str = ""
    last_error: Optional[str] = None

parse_status = ParseStatus()
products_cache: List[ProductChunk] = []

# ------------------ Прокси для картинок ------------------
@app.get("/api/image")
async def proxy_image(url: str = Query(..., description="URL картинки")):
    """Прокси для картинок (обходит CORS)"""
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(url, timeout=15) as resp:
                if resp.status == 200:
                    content = await resp.read()
                    return Response(content=content, media_type=resp.content_type)
                return Response(content=b'', status_code=404)
    except Exception as e:
        print(f"Прокси ошибка {url}: {e}")
        return Response(content=b'', status_code=404)

# ------------------ Парсер ------------------
class PkyarParser:
    def __init__(self, base_url: str = "https://pkyar.ru/"):
        self.base_url = base_url
        self.session = None
        self.products = []

    async def __aenter__(self):
        self.session = aiohttp.ClientSession(headers={
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        return self

    async def __aexit__(self, *args):
        await self.session.close()

    async def fetch_html(self, url: str) -> Optional[str]:
        try:
            async with self.session.get(url, timeout=30) as resp:
                if resp.status == 200:
                    return await resp.text()
        except Exception as e:
            print(f"Ошибка {url}: {e}")
        return None

    def clean_text(self, text: str) -> str:
        if not text:
            return ""
        return re.sub(r'\s+', ' ', text).strip()

    def extract_features(self, soup: BeautifulSoup) -> Dict[str, str]:
        features = {}
        tables = soup.select('table, .props-list, .characteristics')
        for table in tables:
            rows = table.select('tr')
            for row in rows:
                cells = row.select('td, th')
                if len(cells) >= 2:
                    key = self.clean_text(cells[0].get_text())
                    val = self.clean_text(cells[1].get_text())
                    if key and val and len(key) < 50:
                        features[key] = val[:100]
        return features

    async def get_product_links(self) -> List[str]:
        product_links = set()
        sitemap_url = urljoin(self.base_url, "sitemap.xml")
        html = await self.fetch_html(sitemap_url)
        if html:
            try:
                root = ET.fromstring(html)
                ns = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
                for url_elem in root.findall('.//ns:loc', ns):
                    url = url_elem.text
                    if '/production/' in url and url.count('/') >= 4:
                        product_links.add(url)
            except:
                pass
        return list(product_links)

    async def parse_product_page(self, url: str) -> Optional[ProductChunk]:
        html = await self.fetch_html(url)
        if not html:
            return None

        soup = BeautifulSoup(html, 'lxml')

        # Название
        title = None
        for sel in ['h1', '.page-title', '.product-name', '.detail-name']:
            elem = soup.select_one(sel)
            if elem:
                title = self.clean_text(elem.get_text())
                if title and len(title) > 2:
                    break

        if not title:
            return None

        product_id = url.rstrip('/').split('/')[-1]

        # Описание
        description = ""
        for sel in ['.detail-text', '.product-description', '.description', '.content']:
            elem = soup.select_one(sel)
            if elem:
                for tag in elem(['script', 'style']):
                    tag.decompose()
                description = self.clean_text(elem.get_text())
                if description and len(description) > 50:
                    break

        # Характеристики
        features = self.extract_features(soup)

        # Картинка
        image = ""
        for img in soup.select('img[src]'):
            src = img.get('src')
            if src and any(ext in src.lower() for ext in ['.jpg', '.jpeg', '.png']):
                full_url = urljoin(url, src)
                if 'pkyar.ru' in full_url:
                    image = full_url
                    break

        # Текст для чанка
        full_text = f"{title}\n\n{description}\n\n"
        for key, val in list(features.items())[:10]:
            full_text += f"{key}: {val}\n"

        return ProductChunk(
            id=product_id,
            title=title,
            text=full_text[:CHUNK_SIZE],
            image=image,
            features=features
        )

    async def parse_all(self) -> List[ProductChunk]:
        print("🔍 Поиск продукции...")
        product_links = await self.get_product_links()
        total = len(product_links)
        print(f"Найдено товаров: {total}")

        chunks = []
        for idx, url in enumerate(product_links):
            print(f"[{idx+1}/{total}] {url.split('/')[-2]}")
            chunk = await self.parse_product_page(url)
            if chunk:
                chunks.append(chunk)
                print(f"  ✅ {chunk.title[:50]}")
            await asyncio.sleep(0.3)

        return chunks

# ------------------ Запуск парсинга при старте ------------------
@app.on_event("startup")
async def startup_parse():
    global products_cache, parse_status
    print("="*50)
    print("🚀 АВТОПАРСИНГ ЗАПУЩЕН")
    print("="*50)

    try:
        if os.path.exists(PRODUCTION_FILE):
            with open(PRODUCTION_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
                if data.get('chunks') and len(data['chunks']) > 0:
                    products_cache = [ProductChunk(**c) for c in data['chunks']]
                    parse_status.total_products = len(products_cache)
                    parse_status.parsed_products = len(products_cache)
                    parse_status.current_stage = "Готово (кэш)"
                    print(f"✅ Загружено из кэша: {len(products_cache)} товаров")
                    return

        async with PkyarParser() as parser:
            products_cache = await parser.parse_all()
            with open(PRODUCTION_FILE, "w", encoding="utf-8") as f:
                json.dump({"chunks": [c.dict() for c in products_cache]}, f, ensure_ascii=False, indent=2)

            parse_status.total_products = len(products_cache)
            parse_status.parsed_products = len(products_cache)
            parse_status.current_stage = "Готово!"
            print(f"✅ Парсинг завершён! Найдено: {len(products_cache)} товаров")

    except Exception as e:
        parse_status.last_error = str(e)
        print(f"❌ Ошибка: {e}")

# ------------------ API Endpoints ------------------
@app.get("/")
def root():
    return {"message": "Агро-Консультант API", "ready": len(products_cache) > 0, "total": len(products_cache)}

@app.get("/health")
def health():
    return {"status": "ok", "ready": len(products_cache) > 0, "products": len(products_cache)}

@app.get("/api/status")
def get_status():
    return {
        "is_parsing": False,
        "total_products": len(products_cache),
        "parsed_products": len(products_cache),
        "current_stage": parse_status.current_stage,
        "last_error": parse_status.last_error
    }

@app.get("/api/chunks")
def get_chunks():
    return {
        "chunks": [c.dict() for c in products_cache],
        "total": len(products_cache),
        "ready": True
    }

@app.get("/api/search")
def search(q: str):
    if not q:
        return {"results": []}
    q_lower = q.lower()
    results = []
    for c in products_cache:
        if q_lower in c.title.lower() or q_lower in c.text.lower():
            results.append(c.dict())
    return {"results": results[:10]}
'''

# Сохраняем бэкенд
with open("/content/backend_app.py", "w") as f:
    f.write(backend_code)

# ---------- 2) Запуск сервера ----------
import subprocess, time, socket, re

PORT = 8000

# Убиваем предыдущие процессы
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

# Запускаем FastAPI
uvicorn = subprocess.Popen(
    ["python", "-m", "uvicorn", "backend_app:app",
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--app-dir", "/content"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Ждём порт
def wait_port(port):
    for _ in range(30):
        s = socket.socket()
        try:
            s.connect(("127.0.0.1", port))
            s.close()
            return True
        except:
            time.sleep(1)
    return False

wait_port(PORT)

# Запускаем Cloudflare Tunnel
proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel",
     "--url", f"http://127.0.0.1:{PORT}",
     "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Получаем URL
url = None
print("\n⏳ Ожидание публичного URL...")
for _ in range(60):
    line = proc.stdout.readline()
    if line:
        print(line.strip())
    m = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", line)
    if m:
        url = m.group(1)
        break

print("\n" + "="*60)
print("✅ БЭКЕНД ГОТОВ (С ПРОКСИ ДЛЯ КАРТИНОК)")
print("="*60)
print(f"🌐 URL ДЛЯ ПОДКЛЮЧЕНИЯ: {url}")
print("\n📋 Эндпоинты:")
print(f"   {url}/health")
print(f"   {url}/api/status")
print(f"   {url}/api/chunks")
print(f"   {url}/api/image?url=URL_КАРТИНКИ - ПРОКСИ ДЛЯ КАРТИНОК")
print("="*60)

# Проверка прокси
print("\n🧪 Проверка прокси для картинок...")
import requests
test_url = "https://pkyar.ru/bundles/AppBundle/files/catalog/category/scale/2026_02_27_11_06_02_970100.jpeg"
try:
    resp = requests.get(f"{url}/api/image?url={test_url}", timeout=10)
    if resp.status_code == 200:
        print(f"✅ Прокси работает! Статус: {resp.status_code}")
    else:
        print(f"⚠️ Прокси вернул статус: {resp.status_code}")
except Exception as e:
    print(f"⚠️ Ошибка проверки прокси: {e}")

print("\n" + "="*60)
print("🎯 СКОПИРУЙ ЭТОТ URL И ВСТАВЬ В ПОЛЕ CONNECT ВО ФРОНТЕ")
print("="*60)


⏳ Ожидание публичного URL...
2026-04-07T20:44:33Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-07T20:44:33Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-07T20:44:37Z INF +--------------------------------------------------------------------------------------------+
2026-04-07T20:44:37Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-07T20:44:37Z INF |  https://regardless-sala